# Q-Med Colcabamba — Fase 1: ingestión, limpieza y control de calidad

**Problema:** preparar una base reproducible de IPRESS para estudiar la distribución mensual de medicamentos desde **Colcabamba I-4** hacia establecimientos rurales del distrito de Colcabamba, provincia de Tayacaja, Huancavelica.

Esta fase usa tres *snapshots* de [RENIPRESS — SUSALUD](https://datos.susalud.gob.pe/dataset/registro-nacional-de-ipress-renipress). No simula demanda ni optimiza rutas todavía: su producto es una base auditable para las fases siguientes.

### Entradas esperadas

- `data/raw/RENIPRESS_mes_1.csv`
- `data/raw/RENIPRESS_mes_2.csv`
- `data/raw/RENIPRESS_mes_3.csv`

### Salidas

| Archivo | Contenido |
|---|---|
| `data/processed/renipress_longitudinal.csv` | Tres snapshots normalizados y concatenados |
| `data/processed/ipress_colcabamba.csv` | IPRESS activas de Colcabamba en el snapshot más reciente |
| `data/processed/data_quality_report.csv` | Resultado de reglas de calidad por snapshot y sobre la muestra objetivo |
| `data/processed/changes_between_months.csv` | Altas, bajas y cambios de atributos entre snapshots consecutivos |


## 0. Criterios de diseño

1. Los archivos de `data/raw/` son **inmutables**: nunca se sobrescriben.
2. `COD_IPRESS` se lee como texto para conservar ceros iniciales.
3. Cada registro queda trazable mediante `SOURCE_FILE`, `SOURCE_ROW`, `SNAPSHOT_ID` y `SNAPSHOT_ORDER`.
4. Los duplicados se resuelven conservando el registro más completo, pero su cantidad queda registrada.
5. `NORTE` y `ESTE` se convierten a `LATITUD` y `LONGITUD`; valores fuera de los límites plausibles del Perú se marcan, no se inventan ni corrigen silenciosamente.
6. El filtro territorial exige simultáneamente **Huancavelica + Tayacaja + Colcabamba**, evitando confundirlo con otros distritos homónimos del país.


In [1]:
from __future__ import annotations

import hashlib
import os
import re
import unicodedata
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # Permite ejecutar las celdas también como script de validación.
    display = print

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

print(f"pandas: {pd.__version__}")

pandas: 2.3.1


In [2]:
def locate_repo_root(start: Path | None = None) -> Path:
    """Encuentra la raíz de qmed-colcabamba sin depender de cómo se abrió Jupyter."""
    configured_root = os.getenv("QMED_REPO_ROOT")
    if configured_root:
        root = Path(configured_root).expanduser().resolve()
        if (root / "data").is_dir() and (root / "notebooks").is_dir():
            return root
        raise FileNotFoundError(
            "QMED_REPO_ROOT existe, pero no contiene data/ y notebooks/: "
            f"{root}"
        )

    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. Abre Jupyter dentro de "
        "qmed-colcabamba o define la variable QMED_REPO_ROOT."
    )


ROOT = locate_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {ROOT}")
print(f"Datos crudos:       {RAW_DIR}")
print(f"Datos procesados:   {PROCESSED_DIR}")

Raíz del proyecto: O:\Mi unidad\Física - PUCP\2026-1\hackaton\QHub_Logistica_Transporte
Datos crudos:       O:\Mi unidad\Física - PUCP\2026-1\hackaton\QHub_Logistica_Transporte\data\raw
Datos procesados:   O:\Mi unidad\Física - PUCP\2026-1\hackaton\QHub_Logistica_Transporte\data\processed


## 1. Configuración del análisis

Si conoces la fecha exacta de corte de cada archivo, reemplaza los `None` de `SNAPSHOT_DATES` por fechas ISO (`AAAA-MM-DD`). Si no se conoce todavía, el pipeline conserva el orden `mes_1 → mes_2 → mes_3`, deja `SNAPSHOT_DATE` vacío y lo reporta como advertencia; no inventa fechas.

In [3]:
TARGET_DEPARTMENT = "HUANCAVELICA"
TARGET_PROVINCE = "TAYACAJA"
TARGET_DISTRICT = "COLCABAMBA"
ACTIVE_STATUS = "ACTIVO"

EXPECTED_RAW_FILES = (
    "RENIPRESS_mes_1.csv",
    "RENIPRESS_mes_2.csv",
    "RENIPRESS_mes_3.csv",
)
EXPECTED_SNAPSHOT_COUNT = 3

# Completar cuando se verifique la fecha de corte de cada descarga.
SNAPSHOT_DATES: dict[str, str | None] = {
    "RENIPRESS_mes_1.csv": None,
    "RENIPRESS_mes_2.csv": None,
    "RENIPRESS_mes_3.csv": None,
}

# Límites amplios de validación geográfica; no se usan para imputar valores.
PERU_LATITUDE_BOUNDS = (-18.6, 0.6)
PERU_LONGITUDE_BOUNDS = (-82.1, -68.0)

REQUIRED_COLUMNS = {
    "COD_IPRESS",
    "DEPARTAMENTO",
    "PROVINCIA",
    "DISTRITO",
    "ESTADO",
    "NORTE",
    "ESTE",
}

TEXT_COLUMNS = (
    "INSTITUCION",
    "NOMBRE",
    "CLASIFICACION",
    "TIPO_ESTABLECIMIENTO",
    "DEPARTAMENTO",
    "PROVINCIA",
    "DISTRITO",
    "DIRECCION",
    "COD_RED",
    "RED",
    "COD_MICRORED",
    "MICRORED",
    "CATEGORIA",
    "ESTADO",
    "CONDICION",
)

CHANGE_FIELDS = (
    "NOMBRE",
    "INSTITUCION",
    "CLASIFICACION",
    "TIPO_ESTABLECIMIENTO",
    "CATEGORIA",
    "ESTADO",
    "DIRECCION",
    "NORTE",
    "ESTE",
)

COLUMN_ALIASES = {
    "COD_IPRESS": ("CODIGO_IPRESS", "CODIGO_UNICO_IPRESS", "CODIPRESS"),
    "DEPARTAMENTO": ("REGION", "DPTO", "DEPARTAMENTO_IPRESS"),
    "PROVINCIA": ("PROVINCIA_IPRESS",),
    "DISTRITO": ("DISTRITO_IPRESS",),
    "ESTADO": ("ESTADO_REGISTRO", "ESTADO_IPRESS"),
    "NORTE": ("LATITUD", "COORDENADA_NORTE"),
    "ESTE": ("LONGITUD", "COORDENADA_ESTE"),
    "CATEGORIA": ("CATEGORIA_IPRESS",),
    "NOMBRE": ("NOMBRE_IPRESS", "NOM_IPRESS"),
}

print(
    f"Ámbito: {TARGET_DISTRICT}, {TARGET_PROVINCE}, {TARGET_DEPARTMENT} | "
    f"estado: {ACTIVE_STATUS}"
)

Ámbito: COLCABAMBA, TAYACAJA, HUANCAVELICA | estado: ACTIVO


## 2. Funciones de ingestión y normalización

La lectura intenta primero `utf-8-sig`, que es el formato previsto. El *fallback* a `utf-8` o `latin-1` solo evita que una variación de codificación detenga el pipeline y queda documentado en el manifiesto.

In [4]:
def natural_key(value: str | Path) -> list[object]:
    """Orden natural: mes_2 aparece antes que mes_10."""
    return [
        int(token) if token.isdigit() else token.lower()
        for token in re.split(r"(\d+)", str(value))
    ]


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_obj:
        while chunk := file_obj.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_column_name(name: object) -> str:
    text = unicodedata.normalize("NFKD", str(name))
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^0-9A-Za-z]+", "_", text.strip().upper())
    return text.strip("_")


def canonical_key(value: object) -> str | None:
    """Clave comparable: mayúsculas, sin tildes y con espacios consolidados."""
    if pd.isna(value):
        return None
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"\s+", " ", text.replace("\u00a0", " ").strip().upper())
    return text or None


def normalize_text(value: object) -> object:
    if pd.isna(value):
        return pd.NA
    text = unicodedata.normalize("NFKC", str(value)).replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip().upper()
    if text in {"", "NAN", "NONE", "NULL", "<NA>"}:
        return pd.NA
    return text


def parse_decimal(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")


def clean_ipress_code(series: pd.Series) -> pd.Series:
    codes = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    numeric_mask = codes.str.fullmatch(r"\d+", na=False)
    codes.loc[numeric_mask] = codes.loc[numeric_mask].str.zfill(8)
    return codes.replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})


def read_renipress_csv(path: Path) -> tuple[pd.DataFrame, str]:
    errors: list[str] = []
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            frame = pd.read_csv(
                path,
                sep=";",
                encoding=encoding,
                dtype="string",
                low_memory=False,
            )
            if frame.shape[1] == 1:
                raise ValueError("solo se detectó una columna; revisar el separador")
            return frame, encoding
        except (UnicodeDecodeError, pd.errors.ParserError, ValueError) as exc:
            errors.append(f"{encoding}: {exc}")

    details = " | ".join(errors)
    raise ValueError(f"No se pudo leer {path.name}. Intentos: {details}")


def canonicalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    normalized_names = [normalize_column_name(column) for column in frame.columns]
    duplicates = pd.Index(normalized_names)[pd.Index(normalized_names).duplicated()].unique()
    if len(duplicates):
        raise ValueError(
            "La normalización produce columnas duplicadas: "
            f"{duplicates.tolist()}"
        )

    result = frame.copy()
    result.columns = normalized_names

    for canonical, aliases in COLUMN_ALIASES.items():
        if canonical in result.columns:
            continue
        candidates = [
            normalize_column_name(alias)
            for alias in aliases
            if normalize_column_name(alias) in result.columns
        ]
        if len(candidates) == 1:
            result = result.rename(columns={candidates[0]: canonical})
        elif len(candidates) > 1:
            raise ValueError(
                f"Hay más de una columna candidata para {canonical}: {candidates}"
            )

    missing = sorted(REQUIRED_COLUMNS - set(result.columns))
    if missing:
        raise KeyError(
            "Faltan columnas requeridas después de normalizar: "
            f"{missing}. Columnas disponibles: {result.columns.tolist()}"
        )
    return result

In [5]:
def resolve_snapshot_date(path: Path):
    configured = SNAPSHOT_DATES.get(path.name)
    if configured:
        return pd.to_datetime(configured, errors="raise")

    # Admite nombres como RENIPRESS_2026-05-31.csv o RENIPRESS_20260531.csv.
    match = re.search(r"(20\d{2})[-_]?([01]\d)[-_]?([0-3]\d)", path.stem)
    if match:
        return pd.to_datetime("-".join(match.groups()), errors="raise")
    return pd.NaT


def build_snapshot_id(path: Path, order: int) -> str:
    match = re.search(r"MES[_\- ]?(\d+)", path.stem, flags=re.IGNORECASE)
    return f"MES_{int(match.group(1))}" if match else f"SNAPSHOT_{order}"


def deduplicate_snapshot(frame: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Conserva la fila más completa por COD_IPRESS; no colapsa códigos nulos."""
    with_code = frame.loc[frame["COD_IPRESS"].notna()].copy()
    without_code = frame.loc[frame["COD_IPRESS"].isna()].copy()

    duplicate_rows = int(with_code.duplicated("COD_IPRESS", keep=False).sum())
    with_code = (
        with_code.sort_values(
            ["COMPLETENESS_SCORE", "SOURCE_ROW"],
            ascending=[False, True],
            kind="stable",
        )
        .drop_duplicates("COD_IPRESS", keep="first")
    )

    result = (
        pd.concat([with_code, without_code], ignore_index=True)
        .sort_values("SOURCE_ROW", kind="stable")
        .reset_index(drop=True)
    )
    removed = len(frame) - len(result)
    assert removed <= duplicate_rows
    return result, removed


def normalize_snapshot(
    raw_frame: pd.DataFrame,
    path: Path,
    order: int,
    encoding: str,
) -> tuple[pd.DataFrame, dict[str, object]]:
    frame = canonicalize_columns(raw_frame)
    frame.insert(0, "SOURCE_ROW", np.arange(2, len(frame) + 2, dtype=int))
    frame.insert(0, "SOURCE_FILE", path.name)
    frame.insert(0, "SNAPSHOT_ORDER", order)
    frame.insert(0, "SNAPSHOT_ID", build_snapshot_id(path, order))
    frame.insert(0, "SNAPSHOT_DATE", resolve_snapshot_date(path))

    frame["COD_IPRESS"] = clean_ipress_code(frame["COD_IPRESS"])
    frame["COD_IPRESS_VALID"] = frame["COD_IPRESS"].str.fullmatch(
        r"\d{8}", na=False
    )

    for column in set(TEXT_COLUMNS).intersection(frame.columns):
        frame[column] = frame[column].map(normalize_text).astype("string")

    frame["LATITUD"] = parse_decimal(frame["NORTE"])
    frame["LONGITUD"] = parse_decimal(frame["ESTE"])
    frame["COORD_VALID_PERU"] = (
        frame["LATITUD"].between(*PERU_LATITUDE_BOUNDS, inclusive="both")
        & frame["LONGITUD"].between(*PERU_LONGITUDE_BOUNDS, inclusive="both")
    )

    coordinate_candidates = frame[["LATITUD", "LONGITUD"]].notna().all(axis=1)
    if coordinate_candidates.any():
        direct_rate = frame.loc[coordinate_candidates, "COORD_VALID_PERU"].mean()
        inverse_rate = (
            frame.loc[coordinate_candidates, "LONGITUD"].between(
                *PERU_LATITUDE_BOUNDS, inclusive="both"
            )
            & frame.loc[coordinate_candidates, "LATITUD"].between(
                *PERU_LONGITUDE_BOUNDS, inclusive="both"
            )
        ).mean()
        if direct_rate < 0.20 and inverse_rate > 0.80:
            raise ValueError(
                f"{path.name}: NORTE y ESTE parecen estar invertidos. "
                "Revisar antes de continuar; el pipeline no los intercambia en silencio."
            )

    completeness_columns = [
        column
        for column in (
            "COD_IPRESS",
            "NOMBRE",
            "INSTITUCION",
            "DEPARTAMENTO",
            "PROVINCIA",
            "DISTRITO",
            "ESTADO",
            "CATEGORIA",
            "DIRECCION",
            "NORTE",
            "ESTE",
        )
        if column in frame.columns
    ]
    frame["COMPLETENESS_SCORE"] = frame[completeness_columns].notna().sum(axis=1)

    rows_before = len(frame)
    duplicate_key_rows = int(
        frame.loc[frame["COD_IPRESS"].notna()].duplicated(
            "COD_IPRESS", keep=False
        ).sum()
    )
    frame, rows_removed = deduplicate_snapshot(frame)

    audit = {
        "snapshot_id": frame["SNAPSHOT_ID"].iloc[0] if len(frame) else build_snapshot_id(path, order),
        "snapshot_order": order,
        "snapshot_date": resolve_snapshot_date(path),
        "source_file": path.name,
        "encoding": encoding,
        "sha256": sha256_file(path),
        "rows_raw": rows_before,
        "rows_clean": len(frame),
        "duplicate_key_rows": duplicate_key_rows,
        "duplicate_rows_removed": rows_removed,
        "missing_codes": int(frame["COD_IPRESS"].isna().sum()),
        "invalid_codes": int(
            (frame["COD_IPRESS"].notna() & ~frame["COD_IPRESS_VALID"]).sum()
        ),
        "missing_coordinate_pairs": int(
            (~frame[["LATITUD", "LONGITUD"]].notna().all(axis=1)).sum()
        ),
        "invalid_coordinate_pairs": int(
            (
                frame[["LATITUD", "LONGITUD"]].notna().all(axis=1)
                & ~frame["COORD_VALID_PERU"]
            ).sum()
        ),
    }
    return frame, audit

> **Nota técnica:** los límites solo validan coordenadas en grados decimales. Si `NORTE` y `ESTE` vinieran en UTM, se debe confirmar la zona y el sistema de referencia antes de convertirlos; asumirlo automáticamente podría mover una posta cientos de kilómetros.

## 3. Descubrimiento y manifiesto de archivos crudos

In [6]:
expected_paths = [RAW_DIR / filename for filename in EXPECTED_RAW_FILES]
if all(path.exists() for path in expected_paths):
    raw_paths = expected_paths
else:
    raw_paths = sorted(RAW_DIR.glob("RENIPRESS*.csv"), key=natural_key)

if len(raw_paths) != EXPECTED_SNAPSHOT_COUNT:
    discovered = [path.name for path in raw_paths]
    raise FileNotFoundError(
        f"Se esperaban {EXPECTED_SNAPSHOT_COUNT} snapshots en {RAW_DIR}, "
        f"pero se encontraron {len(raw_paths)}: {discovered}"
    )

display(
    pd.DataFrame(
        {
            "SNAPSHOT_ORDER": range(1, len(raw_paths) + 1),
            "SOURCE_FILE": [path.name for path in raw_paths],
            "SIZE_MB": [path.stat().st_size / 1_000_000 for path in raw_paths],
            "CONFIGURED_DATE": [SNAPSHOT_DATES.get(path.name) for path in raw_paths],
        }
    )
)

,SNAPSHOT_ORDER,SOURCE_FILE,SIZE_MB,CONFIGURED_DATE
0,1,RENIPRESS_mes_1.csv,19.075,None
1,2,RENIPRESS_mes_2.csv,19.470,None
2,3,RENIPRESS_mes_3.csv,19.581,None


## 4. Lectura y construcción de la base longitudinal

In [7]:
snapshots: list[pd.DataFrame] = []
audit_records: list[dict[str, object]] = []

for order, path in enumerate(raw_paths, start=1):
    raw_frame, encoding = read_renipress_csv(path)
    clean_frame, audit = normalize_snapshot(raw_frame, path, order, encoding)
    snapshots.append(clean_frame)
    audit_records.append(audit)
    print(
        f"✓ {path.name}: {len(raw_frame):,} filas crudas → "
        f"{len(clean_frame):,} filas limpias"
    )

renipress_longitudinal = (
    pd.concat(snapshots, ignore_index=True, sort=False)
    .sort_values(["SNAPSHOT_ORDER", "COD_IPRESS"], kind="stable", na_position="last")
    .reset_index(drop=True)
)
audit_df = pd.DataFrame(audit_records).sort_values("snapshot_order")

duplicate_after_cleaning = renipress_longitudinal.loc[
    renipress_longitudinal["COD_IPRESS"].notna()
].duplicated(["SNAPSHOT_ID", "COD_IPRESS"], keep=False)
assert not duplicate_after_cleaning.any(), (
    "Persisten claves duplicadas SNAPSHOT_ID + COD_IPRESS después de limpiar."
)

display(audit_df)
display(renipress_longitudinal.head(3))
print(f"Base longitudinal: {len(renipress_longitudinal):,} filas")

✓ RENIPRESS_mes_1.csv: 35,471 filas crudas → 35,471 filas limpias
✓ RENIPRESS_mes_2.csv: 35,581 filas crudas → 35,581 filas limpias
✓ RENIPRESS_mes_3.csv: 35,743 filas crudas → 35,743 filas limpias


,snapshot_id,snapshot_order,snapshot_date,source_file,encoding,sha256,rows_raw,rows_clean,duplicate_key_rows,duplicate_rows_removed,missing_codes,invalid_codes,missing_coordinate_pairs,invalid_coordinate_pairs
0,MES_1,1,NaT,RENIPRESS_mes_1.csv,utf-8-sig,a65c7a8458794b4073f6b2ad8ebafb5749ac6036844ab3e464df573beeb60972,35471,35471,0,0,0,0,13147,7
1,MES_2,2,NaT,RENIPRESS_mes_2.csv,utf-8-sig,1ce93e996ab2f87e9126a34c496b8d8edc0d57e778a51d48da613a52a2e7ddec,35581,35581,0,0,0,0,13154,7
2,MES_3,3,NaT,RENIPRESS_mes_3.csv,utf-8-sig,5cfdad3494440e23d061c71b6feee3494e8341b80378033ea5fea12d09644a40,35743,35743,0,0,0,0,13168,7


,SNAPSHOT_DATE,SNAPSHOT_ID,SNAPSHOT_ORDER,SOURCE_FILE,SOURCE_ROW,INSTITUCION,COD_IPRESS,NOMBRE,CLASIFICACION,TIPO_ESTABLECIMIENTO,DEPARTAMENTO,PROVINCIA,DISTRITO,UBIGEO,DIRECCION,CO_DISA,COD_RED,COD_MICRORRED,DISA,RED,MICRORED,COD_UE,UNIDAD_EJECUTORA,CATEGORIA,TELEFONO,HORARIO,INICIO_ACTIVIDAD,ESTADO,NORTE,ESTE,IMAGEN_1,FE_ACT_IMAGEN_1,IMAGEN_2,FE_ACT_IMAGEN_2,IMAGEN_3,FE_ACT_IMAGEN_3,COD_IPRESS_VALID,LATITUD,LONGITUD,COORD_VALID_PERU,COMPLETENESS_SCORE
0,NaT,MES_1,1,RENIPRESS_mes_1.csv,21163,MINSA,00000001,"HOSPITAL IQUITOS ""CESAR GARAYAR GARCÍA""",HOSPITALES O CLINICAS DE ATENCION GENERAL,ESTABLECIMIENTO DE SALUD CON INTERNAMIENTO,LORETO,MAYNAS,IQUITOS,160101,GRAU CON LIBERTAD,24.0,0.0,0.0,LORETO,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,872.0,HOSPITAL DE APOYO IQUITOS,II-2,958874946,24 HORAS,1960-01-01,ACTIVO,-3.7616779,-73.2538901,http://app20.susalud.gob.pe:8080/registro-renipress-webapp/ipress.htm?action=downloadFile&idArch...,<NA>,http://app20.susalud.gob.pe:8080/registro-renipress-webapp/ipress.htm?action=downloadFile&idArch...,<NA>,http://app20.susalud.gob.pe:8080/registro-renipress-webapp/ipress.htm?action=downloadFile&idArch...,<NA>,True,-3.762,-73.254,True,11
1,NaT,MES_1,1,RENIPRESS_mes_1.csv,1721,GOBIERNO REGIONAL,00000002,CENTRO REHABILITACION ENFERMO MENTAL,CENTROS DE ATENCION PARA DEPENDIENTES A SUSTANCIAS PSICOACTIVAS Y OTRAS DEPENDENCIAS,SERVICIO MÉDICO DE APOYO,LORETO,MAYNAS,SAN JUAN BAUTISTA,160113,CALLE 3 DE MAYO S/N CPM QUILCATACTA,<NA>,0.0,0.0,<NA>,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,870.0,SALUD LORETO,0,ACTUALIZAR,7:30 A 13:00,1900-01-01,BAJA DEFINITIVA,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,9
2,NaT,MES_1,1,RENIPRESS_mes_1.csv,27400,GOBIERNO REGIONAL,00000003,"HOSPITAL REGIONAL DE LORETO ""FELIPE SANTIAGO ARRIOLA IGLESIAS""",HOSPITALES O CLINICAS DE ATENCION ESPECIALIZADA,ESTABLECIMIENTO DE SALUD CON INTERNAMIENTO,LORETO,MAYNAS,PUNCHANA,160108,AVENIDA 28 DE JULIO S/N,24.0,0.0,0.0,LORETO,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,874.0,HOSPITAL REGIONAL LORETO,III-1,(51)065-252746,24 HORAS,1989-07-10,ACTIVO,-3.72696048,-73.25341698,http://app20.susalud.gob.pe:8080/registro-renipress-webapp/ipress.htm?action=downloadFile&idArch...,<NA>,http://app20.susalud.gob.pe:8080/registro-renipress-webapp/ipress.htm?action=downloadFile&idArch...,<NA>,http://app20.susalud.gob.pe:8080/registro-renipress-webapp/ipress.htm?action=downloadFile&idArch...,<NA>,True,-3.727,-73.253,True,11


Base longitudinal: 106,795 filas


## 5. Filtro territorial y snapshot operativo

`ipress_colcabamba` representa el universo utilizable por las fases 2–5: establecimientos **activos** del distrito en el snapshot más reciente. El historial territorial, en cambio, conserva todos los estados para detectar altas, bajas o recategorizaciones.

In [8]:
def matches(series: pd.Series, expected: str) -> pd.Series:
    expected_key = canonical_key(expected)
    return series.map(canonical_key).eq(expected_key).fillna(False)


territorial_mask = (
    matches(renipress_longitudinal["DEPARTAMENTO"], TARGET_DEPARTMENT)
    & matches(renipress_longitudinal["PROVINCIA"], TARGET_PROVINCE)
    & matches(renipress_longitudinal["DISTRITO"], TARGET_DISTRICT)
)
target_history = renipress_longitudinal.loc[territorial_mask].copy()

latest_order = int(renipress_longitudinal["SNAPSHOT_ORDER"].max())
latest_snapshot_id = renipress_longitudinal.loc[
    renipress_longitudinal["SNAPSHOT_ORDER"].eq(latest_order), "SNAPSHOT_ID"
].iloc[0]

ipress_colcabamba = target_history.loc[
    target_history["SNAPSHOT_ORDER"].eq(latest_order)
    & matches(target_history["ESTADO"], ACTIVE_STATUS)
].copy()
ipress_colcabamba = ipress_colcabamba.sort_values(
    ["COD_IPRESS"], kind="stable", na_position="last"
).reset_index(drop=True)

if ipress_colcabamba.empty:
    raise ValueError(
        "El filtro no encontró IPRESS activas en Colcabamba–Tayacaja–Huancavelica. "
        "Revisar valores de DEPARTAMENTO, PROVINCIA, DISTRITO y ESTADO."
    )

summary_by_snapshot = (
    target_history.groupby(["SNAPSHOT_ORDER", "SNAPSHOT_ID"], dropna=False)
    .agg(
        IPRESS_TOTAL=("COD_IPRESS", "nunique"),
        IPRESS_ACTIVAS=("ESTADO", lambda values: matches(values, ACTIVE_STATUS).sum()),
        COORDENADAS_VALIDAS=("COORD_VALID_PERU", "sum"),
    )
    .reset_index()
)

print(f"Snapshot operativo: {latest_snapshot_id}")
display(summary_by_snapshot)

preview_columns = [
    column
    for column in (
        "COD_IPRESS",
        "NOMBRE",
        "CATEGORIA",
        "ESTADO",
        "LATITUD",
        "LONGITUD",
        "COORD_VALID_PERU",
    )
    if column in ipress_colcabamba.columns
]
display(ipress_colcabamba[preview_columns])

Snapshot operativo: MES_3


,SNAPSHOT_ORDER,SNAPSHOT_ID,IPRESS_TOTAL,IPRESS_ACTIVAS,COORDENADAS_VALIDAS
0,1,MES_1,10,10,10
1,2,MES_2,10,10,10
2,3,MES_3,10,10,10


,COD_IPRESS,NOMBRE,CATEGORIA,ESTADO,LATITUD,LONGITUD,COORD_VALID_PERU
0,00004090,COLCABAMBA,I-4,ACTIVO,-12.408,-74.681,True
1,00004092,CARPAPATA,I-2,ACTIVO,-12.388,-74.750,True
2,00004093,POCCYACC,I-1,ACTIVO,-12.478,-74.650,True
3,00004094,OCORO,I-2,ACTIVO,-12.350,-74.728,True
4,00004095,TOCAS,I-2,ACTIVO,-12.443,-74.653,True
5,00004096,TOCCLLACURI,I-1,ACTIVO,-12.347,-74.795,True
6,00007387,CHACHAS,I-1,ACTIVO,-12.428,-74.619,True
7,00007450,RUNDOVILCA,I-1,ACTIVO,-12.324,-74.782,True
8,00012900,RANRA,I-1,ACTIVO,-12.341,-74.743,True
9,00025329,POSTA MEDICA CAMPO ARMIÑO,I-2,ACTIVO,-12.967,-75.574,True


### Verificación del posible centro de distribución

Todavía no se fuerza un código de origen. Se muestran candidatos cuyo nombre contiene `COLCABAMBA` o cuya categoría es `I-4`; la selección definitiva se valida en la Fase 2 con nombre, categoría y coordenadas.

In [9]:
name_mask = (
    ipress_colcabamba["NOMBRE"].str.contains("COLCABAMBA", case=False, na=False)
    if "NOMBRE" in ipress_colcabamba.columns
    else pd.Series(False, index=ipress_colcabamba.index)
)
category_mask = (
    matches(ipress_colcabamba["CATEGORIA"], "I-4")
    if "CATEGORIA" in ipress_colcabamba.columns
    else pd.Series(False, index=ipress_colcabamba.index)
)

depot_candidates = ipress_colcabamba.loc[name_mask | category_mask, preview_columns]
display(depot_candidates)

if depot_candidates.empty:
    print("ADVERTENCIA: no se identificó automáticamente un candidato para Colcabamba I-4.")

,COD_IPRESS,NOMBRE,CATEGORIA,ESTADO,LATITUD,LONGITUD,COORD_VALID_PERU
0,00004090,COLCABAMBA,I-4,ACTIVO,-12.408,-74.681,True


## 6. Cambios entre snapshots

La comparación se restringe al ámbito territorial del proyecto. Se registran tres tipos de evento:

- `ALTA`: el código aparece por primera vez.
- `BAJA_DEL_SNAPSHOT`: el código deja de aparecer en el siguiente corte; no equivale automáticamente a baja administrativa.
- `CAMBIO`: cambia uno de los campos vigilados.

In [10]:
def comparable_value(value: object) -> object:
    if pd.isna(value):
        return None
    if isinstance(value, (float, np.floating)):
        return round(float(value), 6)
    return canonical_key(value)


def detect_snapshot_changes(
    history: pd.DataFrame,
    fields: Iterable[str],
) -> pd.DataFrame:
    event_columns = [
        "FROM_SNAPSHOT",
        "TO_SNAPSHOT",
        "COD_IPRESS",
        "EVENT_TYPE",
        "FIELD",
        "OLD_VALUE",
        "NEW_VALUE",
    ]
    events: list[dict[str, object]] = []
    fields = [field for field in fields if field in history.columns]
    orders = sorted(history["SNAPSHOT_ORDER"].dropna().astype(int).unique())

    for previous_order, current_order in zip(orders, orders[1:]):
        previous = history.loc[
            history["SNAPSHOT_ORDER"].eq(previous_order)
            & history["COD_IPRESS"].notna()
        ].set_index("COD_IPRESS")
        current = history.loc[
            history["SNAPSHOT_ORDER"].eq(current_order)
            & history["COD_IPRESS"].notna()
        ].set_index("COD_IPRESS")

        previous_id = previous["SNAPSHOT_ID"].iloc[0] if len(previous) else f"SNAPSHOT_{previous_order}"
        current_id = current["SNAPSHOT_ID"].iloc[0] if len(current) else f"SNAPSHOT_{current_order}"

        previous_codes = set(previous.index)
        current_codes = set(current.index)

        for code in sorted(current_codes - previous_codes):
            events.append(
                {
                    "FROM_SNAPSHOT": previous_id,
                    "TO_SNAPSHOT": current_id,
                    "COD_IPRESS": code,
                    "EVENT_TYPE": "ALTA",
                    "FIELD": pd.NA,
                    "OLD_VALUE": pd.NA,
                    "NEW_VALUE": pd.NA,
                }
            )

        for code in sorted(previous_codes - current_codes):
            events.append(
                {
                    "FROM_SNAPSHOT": previous_id,
                    "TO_SNAPSHOT": current_id,
                    "COD_IPRESS": code,
                    "EVENT_TYPE": "BAJA_DEL_SNAPSHOT",
                    "FIELD": pd.NA,
                    "OLD_VALUE": pd.NA,
                    "NEW_VALUE": pd.NA,
                }
            )

        for code in sorted(previous_codes & current_codes):
            for field in fields:
                old_value = previous.at[code, field]
                new_value = current.at[code, field]
                if comparable_value(old_value) != comparable_value(new_value):
                    events.append(
                        {
                            "FROM_SNAPSHOT": previous_id,
                            "TO_SNAPSHOT": current_id,
                            "COD_IPRESS": code,
                            "EVENT_TYPE": "CAMBIO",
                            "FIELD": field,
                            "OLD_VALUE": old_value,
                            "NEW_VALUE": new_value,
                        }
                    )

    return pd.DataFrame(events, columns=event_columns)


changes_between_months = detect_snapshot_changes(target_history, CHANGE_FIELDS)
print(f"Eventos detectados: {len(changes_between_months):,}")
display(changes_between_months.head(20))

Eventos detectados: 1


,FROM_SNAPSHOT,TO_SNAPSHOT,COD_IPRESS,EVENT_TYPE,FIELD,OLD_VALUE,NEW_VALUE
0,MES_1,MES_2,00007450,CAMBIO,INSTITUCION,MINSA,GOBIERNO REGIONAL


## 7. Reporte de calidad

Cada regla informa filas afectadas, denominador, porcentaje y severidad. `FAIL` señala un problema que debe resolverse antes de optimizar; `WARN` documenta una limitación que no necesariamente impide continuar.

In [11]:
def quality_row(
    scope: str,
    rule: str,
    severity: str,
    affected_rows: int,
    total_rows: int,
    description: str,
) -> dict[str, object]:
    percentage = 100 * affected_rows / total_rows if total_rows else np.nan
    if affected_rows == 0:
        status = "PASS"
    else:
        status = "FAIL" if severity == "ERROR" else "WARN"
    return {
        "SCOPE": scope,
        "RULE": rule,
        "SEVERITY": severity,
        "STATUS": status,
        "AFFECTED_ROWS": int(affected_rows),
        "TOTAL_ROWS": int(total_rows),
        "PERCENTAGE": percentage,
        "DESCRIPTION": description,
    }


quality_records: list[dict[str, object]] = []
for audit in audit_records:
    scope = str(audit["snapshot_id"])
    total = int(audit["rows_clean"])
    quality_records.extend(
        [
            quality_row(
                scope,
                "MISSING_COD_IPRESS",
                "ERROR",
                int(audit["missing_codes"]),
                total,
                "COD_IPRESS vacío después de la limpieza.",
            ),
            quality_row(
                scope,
                "INVALID_COD_IPRESS",
                "ERROR",
                int(audit["invalid_codes"]),
                total,
                "El código no tiene ocho dígitos.",
            ),
            quality_row(
                scope,
                "DUPLICATES_REMOVED",
                "WARNING",
                int(audit["duplicate_rows_removed"]),
                int(audit["rows_raw"]),
                "Filas duplicadas eliminadas conservando el registro más completo.",
            ),
            quality_row(
                scope,
                "MISSING_COORDINATE_PAIR",
                "WARNING",
                int(audit["missing_coordinate_pairs"]),
                total,
                "Falta latitud, longitud o ambas.",
            ),
            quality_row(
                scope,
                "COORDINATES_OUTSIDE_PERU",
                "ERROR",
                int(audit["invalid_coordinate_pairs"]),
                total,
                "Par numérico fuera de los límites geográficos plausibles del Perú.",
            ),
            quality_row(
                scope,
                "MISSING_SNAPSHOT_DATE",
                "WARNING",
                total if pd.isna(audit["snapshot_date"]) else 0,
                total,
                "No se consignó la fecha exacta del corte; se conserva el orden relativo.",
            ),
        ]
    )

target_total = len(ipress_colcabamba)
target_invalid_coordinates = int((~ipress_colcabamba["COORD_VALID_PERU"]).sum())
target_duplicate_codes = int(
    ipress_colcabamba["COD_IPRESS"].duplicated(keep=False).sum()
)
quality_records.extend(
    [
        quality_row(
            "LATEST_COLCABAMBA_ACTIVE",
            "NO_ACTIVE_TARGET_IPRESS",
            "ERROR",
            int(target_total == 0),
            1,
            "El snapshot operativo debe contener al menos una IPRESS activa.",
        ),
        quality_row(
            "LATEST_COLCABAMBA_ACTIVE",
            "DUPLICATE_TARGET_CODES",
            "ERROR",
            target_duplicate_codes,
            target_total,
            "COD_IPRESS debe ser único en la muestra operativa.",
        ),
        quality_row(
            "LATEST_COLCABAMBA_ACTIVE",
            "TARGET_WITHOUT_VALID_COORDINATES",
            "ERROR",
            target_invalid_coordinates,
            target_total,
            "Toda IPRESS que entre al modelo de rutas necesita coordenadas válidas.",
        ),
        quality_row(
            "LATEST_COLCABAMBA_ACTIVE",
            "DEPOT_CANDIDATE_NOT_UNIQUE",
            "WARNING",
            int(len(depot_candidates) != 1),
            1,
            "Debe verificarse manualmente un único origen Colcabamba I-4.",
        ),
    ]
)

data_quality_report = pd.DataFrame(quality_records)
display(data_quality_report)

critical_failures = data_quality_report.query(
    "SEVERITY == 'ERROR' and STATUS == 'FAIL'"
)
print(f"Reglas críticas incumplidas: {len(critical_failures)}")

,SCOPE,RULE,SEVERITY,STATUS,AFFECTED_ROWS,TOTAL_ROWS,PERCENTAGE,DESCRIPTION
0,MES_1,MISSING_COD_IPRESS,ERROR,PASS,0,35471,0.000,COD_IPRESS vacío después de la limpieza.
1,MES_1,INVALID_COD_IPRESS,ERROR,PASS,0,35471,0.000,El código no tiene ocho dígitos.
2,MES_1,DUPLICATES_REMOVED,WARNING,PASS,0,35471,0.000,Filas duplicadas eliminadas conservando el registro más completo.
3,MES_1,MISSING_COORDINATE_PAIR,WARNING,WARN,13147,35471,37.064,"Falta latitud, longitud o ambas."
4,MES_1,COORDINATES_OUTSIDE_PERU,ERROR,FAIL,7,35471,0.020,Par numérico fuera de los límites geográficos plausibles del Perú.
5,MES_1,MISSING_SNAPSHOT_DATE,WARNING,WARN,35471,35471,100.000,No se consignó la fecha exacta del corte; se conserva el orden relativo.
6,MES_2,MISSING_COD_IPRESS,ERROR,PASS,0,35581,0.000,COD_IPRESS vacío después de la limpieza.
7,MES_2,INVALID_COD_IPRESS,ERROR,PASS,0,35581,0.000,El código no tiene ocho dígitos.
8,MES_2,DUPLICATES_REMOVED,WARNING,PASS,0,35581,0.000,Filas duplicadas eliminadas conservando el registro más completo.
9,MES_2,MISSING_COORDINATE_PAIR,WARNING,WARN,13154,35581,36.969,"Falta latitud, longitud o ambas."


Reglas críticas incumplidas: 3


## 8. Exportación reproducible

Los CSV se guardan con `utf-8-sig` para mantener tildes y facilitar su apertura en Excel. Solo se escriben archivos dentro de `data/processed/`.

In [12]:
OUTPUTS = {
    "renipress_longitudinal.csv": renipress_longitudinal,
    "ipress_colcabamba.csv": ipress_colcabamba,
    "data_quality_report.csv": data_quality_report,
    "changes_between_months.csv": changes_between_months,
}

export_summary: list[dict[str, object]] = []
for filename, frame in OUTPUTS.items():
    output_path = PROCESSED_DIR / filename
    frame.to_csv(output_path, index=False, encoding="utf-8-sig")
    export_summary.append(
        {
            "FILE": filename,
            "ROWS": len(frame),
            "COLUMNS": len(frame.columns),
            "SIZE_KB": output_path.stat().st_size / 1_000,
            "PATH": output_path.relative_to(ROOT).as_posix(),
        }
    )

export_summary_df = pd.DataFrame(export_summary)
display(export_summary_df)

,FILE,ROWS,COLUMNS,SIZE_KB,PATH
0,renipress_longitudinal.csv,106795,41,"64,723.772",data/processed/renipress_longitudinal.csv
1,ipress_colcabamba.csv,10,41,8.531,data/processed/ipress_colcabamba.csv
2,data_quality_report.csv,22,8,2.657,data/processed/data_quality_report.csv
3,changes_between_months.csv,1,7,0.143,data/processed/changes_between_months.csv


## 9. Criterios de cierre de la Fase 1

La fase queda técnicamente lista para pasar al EDA geoespacial cuando:

- se cargaron exactamente tres snapshots;
- no quedan duplicados de `SNAPSHOT_ID + COD_IPRESS`;
- existe al menos una IPRESS activa en Colcabamba–Tayacaja–Huancavelica;
- se verificó un único origen **Colcabamba I-4**;
- las IPRESS que participarán en rutas tienen coordenadas válidas;
- las fechas exactas de corte están consignadas o la limitación se declara explícitamente.

Los errores de calidad no se ocultan: los archivos se exportan para inspección, pero **no debe iniciarse la optimización** hasta resolver las reglas críticas.

In [13]:
phase_summary = pd.DataFrame(
    [
        {
            "INDICATOR": "Snapshots cargados",
            "VALUE": len(snapshots),
            "EXPECTED": EXPECTED_SNAPSHOT_COUNT,
            "STATUS": "OK" if len(snapshots) == EXPECTED_SNAPSHOT_COUNT else "REVISAR",
        },
        {
            "INDICATOR": "Filas longitudinales",
            "VALUE": len(renipress_longitudinal),
            "EXPECTED": "> 0",
            "STATUS": "OK" if len(renipress_longitudinal) > 0 else "REVISAR",
        },
        {
            "INDICATOR": "IPRESS activas objetivo",
            "VALUE": len(ipress_colcabamba),
            "EXPECTED": "> 0",
            "STATUS": "OK" if len(ipress_colcabamba) > 0 else "REVISAR",
        },
        {
            "INDICATOR": "Candidatos a origen",
            "VALUE": len(depot_candidates),
            "EXPECTED": 1,
            "STATUS": "OK" if len(depot_candidates) == 1 else "REVISAR",
        },
        {
            "INDICATOR": "Reglas críticas incumplidas",
            "VALUE": len(critical_failures),
            "EXPECTED": 0,
            "STATUS": "OK" if len(critical_failures) == 0 else "REVISAR",
        },
    ]
)

display(phase_summary)

if (phase_summary["STATUS"] == "OK").all():
    print("FASE 1 APROBADA: la base está lista para 02_eda_geospatial.ipynb.")
else:
    print("FASE 1 EJECUTADA CON OBSERVACIONES: revisar los indicadores marcados.")

,INDICATOR,VALUE,EXPECTED,STATUS
0,Snapshots cargados,3,3,OK
1,Filas longitudinales,106795,> 0,OK
2,IPRESS activas objetivo,10,> 0,OK
3,Candidatos a origen,1,1,OK
4,Reglas críticas incumplidas,3,0,REVISAR


FASE 1 EJECUTADA CON OBSERVACIONES: revisar los indicadores marcados.


## Dataset de trabajo

El dataset principal para las siguientes fases será **`ipress_colcabamba.csv`**, que contiene las IPRESS del ámbito geográfico seleccionado y servirá para definir los nodos de distribución.

Los demás archivos cumplen funciones complementarias:

- **`renipress_longitudinal.csv`**: consolida los tres meses de RENIPRESS.
- **`data_quality_report.csv`**: resume problemas de calidad, valores faltantes y duplicados.
- **`changes_between_months.csv`**: identifica variaciones en las IPRESS entre meses.